Lead Qualification & Routing Agent

In [ ]:
# Import the libraries required for environment variables, JSON handling, and the OpenAI client.
import os
import json

from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
# Load the OpenAI API key and initialize the OpenAI client.
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found, kindly recheck this.")
else:
    print("API key found.")



In [ ]:
openai = OpenAI()

In [ ]:
# Define a realistic B2B lead and the instructions for extracting structured information from it.
lead = """
Hi,

I'm the operations manager at a logistics company with about 80 employees.

We're looking for an AI automation system that can process customer
support requests and automatically route them to the right teams.

Our current operation handles around 5,000 support tickets per month.

We have allocated approximately $2,000 per month for the solution.

Can we schedule a demo next week?

Best,
Michael
"""

lead_prompt = """
You are a B2B lead intelligence assistant.

Analyze the lead and return valid JSON using exactly this structure:

{
    "company": "",
    "role": "",
    "industry": "",
    "company_size": "",
    "need": "",
    "budget": "",
    "buying_intent": "",
    "timeline": "",
    "requested_action": ""
}

The buying_intent must be exactly one of:
- high
- medium
- low
- unknown

Only extract information explicitly present in the lead.

Do not invent information.

For unknown information, use an empty string.

Return only valid JSON.
"""

In [ ]:
# Define a realistic B2B lead and the instructions for extracting structured information from it.
lead = """
Hi,

I'm the operations manager at a logistics company with about 80 employees.

We're looking for an AI automation system that can process customer
support requests and automatically route them to the right teams.

Our current operation handles around 5,000 support tickets per month.

We have allocated approximately $2,000 per month for the solution.

Can we schedule a demo next week?

Best,
Michael
"""

lead_prompt = """
You are a B2B lead intelligence assistant.

Analyze the lead and return valid JSON using exactly this structure:

{
    "company": "",
    "role": "",
    "industry": "",
    "company_size": "",
    "need": "",
    "budget": "",
    "buying_intent": "",
    "timeline": "",
    "requested_action": ""
}

The buying_intent must be exactly one of:
- high
- medium
- low
- unknown

Only extract information explicitly present in the lead.

Do not invent information.

For unknown information, use an empty string.

Return only valid JSON.
"""

In [ ]:
# Validate the extracted fields and analyze a lead using one LLM call.
lead_fields = [
    "company",
    "role",
    "industry",
    "company_size",
    "need",
    "budget",
    "buying_intent",
    "timeline",
    "requested_action"
]


def validate_lead(data):
    if not data:
        return False

    return all(field in data for field in lead_fields)


def analyze_lead(lead):
    messages = [
        {"role": "system", "content": lead_prompt},
        {"role": "user", "content": lead}
    ]

    try:
        response = openai.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages
        )

        result = response.choices[0].message.content
        data = json.loads(result)

        if not validate_lead(data):
            print("The LLM response is missing required fields.")
            return None

        return data

    except json.JSONDecodeError:
        print("The LLM returned invalid JSON.")
        return None

    except Exception as error:
        print(f"An error occurred: {error}")
        return None

In [ ]:
# Run the LLM extraction and display the structured lead information.
result = analyze_lead(lead)

for field, value in result.items():
    print(f"{field.upper()}: {value}")

In [ ]:
# Calculate the lead score using buying intent and meaningful commercial signals.
def calculate_lead_score(data):
    score = 0

    intent_scores = {
        "high": 60,
        "medium": 35,
        "low": 10,
        "unknown": 0
    }

    score += intent_scores.get(data["buying_intent"], 0)

    if data["budget"]:
        score += 15

    if data["timeline"]:
        score += 15

    action = data["requested_action"].lower()

    if any(term in action for term in [
        "demo",
        "sales",
        "proposal",
        "purchase",
        "buy"
    ]):
        score += 10

    return score

In [ ]:
# Convert the score into a qualification category and determine the recommended business action.
def qualify_lead(score):
    if score >= 80:
        return "hot"
    elif score >= 50:
        return "warm"
    else:
        return "cold"


def route_lead(qualification):
    if qualification == "hot":
        return "sales_followup"
    elif qualification == "warm":
        return "followup"
    else:
        return "nurture"

In [ ]:
# Combine extraction, validation, scoring, qualification, and routing into one pipeline.
def qualify_and_route(lead):
    data = analyze_lead(lead)

    if not data:
        return None

    score = calculate_lead_score(data)
    qualification = qualify_lead(score)
    action = route_lead(qualification)

    return {
        **data,
        "lead_score": score,
        "qualification": qualification,
        "recommended_action": action
    }


result = qualify_and_route(lead)

for field, value in result.items():
    print(f"{field.upper()}: {value}")

In [ ]:
# Test the scoring and routing system against leads with different buying signals.
edge_case_leads = [
    {
        "name": "High intent, no budget",
        "lead": """
        We want to purchase an AI automation platform for our support team.
        We need to get started next month. Can we schedule a demo?
        """
    },
    {
        "name": "High intent, no timeline",
        "lead": """
        We are ready to purchase your enterprise AI automation platform.
        Our company has allocated $4,000 per month. Can we speak with sales?
        """
    },
    {
        "name": "Low intent, budget mentioned",
        "lead": """
        I am researching AI automation tools for my company.
        We have a $2,000 monthly budget, but we are not currently planning
        to purchase anything.
        """
    },
    {
        "name": "Medium intent, demo request",
        "lead": """
        We are exploring AI automation for our operations team.
        We would like to schedule a demo to understand what your platform offers.
        """
    },
    {
        "name": "Vague lead",
        "lead": """
        Hi, I heard about your company and would like to know more about
        your AI solutions.
        """
    },
    {
        "name": "Large company, weak intent",
        "lead": """
        We have more than 500 employees and are researching automation
        possibilities. We are not currently looking to purchase a solution.
        """
    }
]

for item in edge_case_leads:
    result = qualify_and_route(item["lead"])

    print(f"\nCASE: {item['name']}")
    print(f"Buying intent: {result['buying_intent']}")
    print(f"Budget: {result['budget']}")
    print(f"Timeline: {result['timeline']}")
    print(f"Requested action: {result['requested_action']}")
    print(f"Score: {result['lead_score']}")
    print(f"Qualification: {result['qualification']}")
    print(f"Action: {result['recommended_action']}")
    print("-" * 50)

In [ ]:
# Evaluate the complete qualification pipeline against representative test leads and calculate accuracy.
qualification_tests = [
    {
        "expected": "hot",
        "lead": "We want to purchase your enterprise plan. We have a $5,000 monthly budget and need implementation next month. Please schedule a sales call."
    },
    {
        "expected": "hot",
        "lead": "We are ready to buy an AI automation platform. We need a solution immediately and would like to schedule a demo."
    },
    {
        "expected": "warm",
        "lead": "We are exploring AI automation and have a budget of $2,000 per month. We expect to make a decision later this year."
    },
    {
        "expected": "cold",
        "lead": "I'm researching AI automation tools and would like to understand how your platform works."
    },
    {
        "expected": "cold",
        "lead": "We are not currently planning to purchase an automation solution. I'm just researching the market."
    },
    {
        "expected": "hot",
        "lead": "Our company is interested in purchasing your software. We have a budget and would like to speak with your sales team this week."
    },
    {
        "expected": "cold",
        "lead": "Can you send me information about your AI solutions?"
    },
    {
        "expected": "warm",
        "lead": "We're considering an automation project and have allocated a budget. We'd like to learn more about your platform."
    }
]

correct = 0

for item in qualification_tests:
    result = qualify_and_route(item["lead"])
    predicted = result["qualification"]

    print(f"Expected: {item['expected']}")
    print(f"Predicted: {predicted}")
    print(f"Score: {result['lead_score']}")
    print("-" * 40)

    if predicted == item["expected"]:
        correct += 1

accuracy = correct / len(qualification_tests)

print(f"Qualification Accuracy: {accuracy:.2%}")